In [11]:
import pandas as pd
import numpy as np
import glob
import os
import re
from collections import Counter

# ========================== CONFIGURATION ==========================
input_folder = r"C:\Users\admin\Downloads\Excel\Excel"
output_folder = r"C:\Users\admin\Downloads\Excel\sample\sample output"
file_pattern = "*.xlsx"          # change to "*.csv" if needed

time_keyword = "time"            # column name for time (case‑insensitive)
angle_keyword = "left knee"      # column name for left knee angle (case‑insensitive)

# ========================== HELPER FUNCTIONS ==========================
def compute_derivatives(time_series, angle_series):
    """
    Compute velocity, acceleration, jerk using numpy.gradient.
    Returns arrays of same length.
    """
    vel = np.gradient(angle_series, time_series)
    acc = np.gradient(vel, time_series)
    jerk = np.gradient(acc, time_series)
    return vel, acc, jerk

def compute_stats(series):
    """Return dict of max, min, ROM, mean, median, std, mode."""
    if len(series) == 0:
        return {k: np.nan for k in ['max','min','ROM','mean','median','std','mode']}
    mode_val = series.mode()
    mode_val = mode_val.iloc[0] if not mode_val.empty else np.nan
    return {
        'max': series.max(),
        'min': series.min(),
        'ROM': series.max() - series.min(),
        'mean': series.mean(),
        'median': series.median(),
        'std': series.std(),
        'mode': mode_val
    }

def extract_quality_view(filename):
    """
    Extract quality (good/bad) and view (diagonal/front/side) from filename.
    Returns (quality, view) or (None, None) if not found.
    """
    quality = None
    view = None
    lower = filename.lower()
    if "good" in lower:
        quality = "good"
    elif "bad" in lower:
        quality = "bad"
    else:
        return None, None
    
    if "diagonal" in lower:
        view = "diagonal"
    elif "front" in lower:
        view = "front"
    elif "side" in lower:
        view = "side"
    else:
        return None, None
    
    return quality, view

# ========================== PROCESS FILES ==========================
file_list = glob.glob(os.path.join(input_folder, file_pattern))
print(f"Found {len(file_list)} files.")

# Dictionary: subject_id -> list of trials (each trial is dict)
subject_trials = {}

# Global lists for summary (across all subjects)
global_angle_stats = []
global_deriv_stats = []

for file_path in file_list:
    filename = os.path.basename(file_path)
    
    # Extract subject ID (e.g., "subject_010")
    subject_match = re.search(r"(subject_\d+)", filename)
    if not subject_match:
        print(f"Skipping {filename}: no subject ID")
        continue
    subject_id = subject_match.group(1)
    
    # Extract quality and view
    quality, view = extract_quality_view(filename)
    if quality is None or view is None:
        print(f"Skipping {filename}: quality or view not found")
        continue
    trial_name = f"{quality}_{view}"   # e.g., good_diagonal
    
    # Read file, skip first 10 rows
    try:
        if filename.endswith('.csv'):
            df = pd.read_csv(file_path, skiprows=10)
        else:
            df = pd.read_excel(file_path, skiprows=10)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue
    
    # Find time and angle columns
    time_col = None
    angle_col = None
    for col in df.columns:
        if time_keyword.lower() in col.lower():
            time_col = col
        if angle_keyword.lower() in col.lower():
            angle_col = col
    if time_col is None or angle_col is None:
        print(f"Skipping {filename}: time or angle column missing")
        continue
    
    # Extract time and angle, drop NaNs
    trial_df = df[[time_col, angle_col]].dropna().copy()
    trial_df.rename(columns={time_col: 'Time', angle_col: 'Angle'}, inplace=True)
    
    # Ensure numeric
    trial_df['Time'] = pd.to_numeric(trial_df['Time'], errors='coerce')
    trial_df['Angle'] = pd.to_numeric(trial_df['Angle'], errors='coerce')
    trial_df.dropna(inplace=True)
    
    if len(trial_df) < 3:
        print(f"Skipping {filename}: not enough data points after cleaning")
        continue
    
    # Compute derivatives
    time_vals = trial_df['Time'].values
    angle_vals = trial_df['Angle'].values
    vel, acc, jerk = compute_derivatives(time_vals, angle_vals)
    
    trial_df['Velocity'] = vel
    trial_df['Acceleration'] = acc
    trial_df['Jerk'] = jerk
    
    # Store in subject_trials
    if subject_id not in subject_trials:
        subject_trials[subject_id] = []
    subject_trials[subject_id].append({
        'trial_name': trial_name,
        'data': trial_df
    })
    
    # ---- Compute stats for this trial and add to global lists ----
    angle_stats = compute_stats(trial_df['Angle'])
    vel_stats = compute_stats(trial_df['Velocity'])
    acc_stats = compute_stats(trial_df['Acceleration'])
    jerk_stats = compute_stats(trial_df['Jerk'])
    
    # Angle stats row
    global_angle_stats.append({
        'Subject': subject_id,
        'Video': trial_name,
        'Max': angle_stats['max'],
        'Min': angle_stats['min'],
        'ROM': angle_stats['ROM'],
        'Mean': angle_stats['mean'],
        'Median': angle_stats['median'],
        'Std_Dev': angle_stats['std'],
        'Mode': angle_stats['mode']
    })
    
    # Derivative stats row
    global_deriv_stats.append({
        'Subject': subject_id,
        'Video': trial_name,
        'Vel_Max': vel_stats['max'],
        'Vel_Min': vel_stats['min'],
        'Vel_Mean': vel_stats['mean'],
        'Vel_Median': vel_stats['median'],
        'Vel_Std': vel_stats['std'],
        'Vel_Mode': vel_stats['mode'],
        'Acc_Max': acc_stats['max'],
        'Acc_Min': acc_stats['min'],
        'Acc_Mean': acc_stats['mean'],
        'Acc_Median': acc_stats['median'],
        'Acc_Std': acc_stats['std'],
        'Acc_Mode': acc_stats['mode'],
        'Jerk_Max': jerk_stats['max'],
        'Jerk_Min': jerk_stats['min'],
        'Jerk_Mean': jerk_stats['mean'],
        'Jerk_Median': jerk_stats['median'],
        'Jerk_Std': jerk_stats['std'],
        'Jerk_Mode': jerk_stats['mode']
    })

# ========================== CREATE OUTPUT FILES (per subject) ==========================
os.makedirs(output_folder, exist_ok=True)

for subject_id, trials_list in subject_trials.items():
    print(f"\nProcessing subject: {subject_id} ({len(trials_list)} trials)")
    
    # ---- Check for duplicate trial names (should not happen) ----
    names = [t['trial_name'] for t in trials_list]
    if len(set(names)) < len(names):
        print(f"  Warning: duplicate trial names found for {subject_id}. Appending numbers to ensure uniqueness.")
        count = Counter(names)
        for t in trials_list:
            if count[t['trial_name']] > 1:
                base = t['trial_name']
                i = 1
                while f"{base}_{i}" in names:
                    i += 1
                new_name = f"{base}_{i}"
                t['trial_name'] = new_name
                names.append(new_name)
    
    # ---- Sheet1: Wide time series ----
    max_rows = max(len(t['data']) for t in trials_list)
    index_col = np.arange(1, max_rows+1)
    sheet1_df = pd.DataFrame({'Index': index_col})
    
    for trial in trials_list:
        name = trial['trial_name']
        df_trial = trial['data'].copy()
        if len(df_trial) < max_rows:
            pad = pd.DataFrame(np.nan, index=range(max_rows - len(df_trial)), columns=df_trial.columns)
            df_trial = pd.concat([df_trial, pad], ignore_index=True)
        df_trial.rename(columns={
            'Time': f'{name}_time',
            'Angle': f'{name}_angle',
            'Velocity': f'{name}_velocity',
            'Acceleration': f'{name}_acceleration',
            'Jerk': f'{name}_jerk'
        }, inplace=True)
        sheet1_df = pd.concat([sheet1_df, df_trial], axis=1)
    
    # ---- Sheet2: Angle statistics (per subject) ----
    # We already have global_angle_stats, but we need per-subject sheets as well.
    # We'll recompute from the trials_list for this subject.
    stats_rows = []
    for trial in trials_list:
        name = trial['trial_name']
        angle_series = trial['data']['Angle']
        stats = compute_stats(angle_series)
        stats_rows.append({
            'Video': name,
            'Max': stats['max'],
            'Min': stats['min'],
            'ROM': stats['ROM'],
            'Mean': stats['mean'],
            'Median': stats['median'],
            'Std_Dev': stats['std'],
            'Mode': stats['mode']
        })
    sheet2_df = pd.DataFrame(stats_rows)
    
    # ---- Sheet3: Derivative statistics (per subject) ----
    deriv_stats_rows = []
    for trial in trials_list:
        name = trial['trial_name']
        vel_series = trial['data']['Velocity']
        acc_series = trial['data']['Acceleration']
        jerk_series = trial['data']['Jerk']
        
        vel_stats = compute_stats(vel_series)
        acc_stats = compute_stats(acc_series)
        jerk_stats = compute_stats(jerk_series)
        
        deriv_stats_rows.append({
            'Video': name,
            'Vel_Max': vel_stats['max'],
            'Vel_Min': vel_stats['min'],
            'Vel_Mean': vel_stats['mean'],
            'Vel_Median': vel_stats['median'],
            'Vel_Std': vel_stats['std'],
            'Vel_Mode': vel_stats['mode'],
            'Acc_Max': acc_stats['max'],
            'Acc_Min': acc_stats['min'],
            'Acc_Mean': acc_stats['mean'],
            'Acc_Median': acc_stats['median'],
            'Acc_Std': acc_stats['std'],
            'Acc_Mode': acc_stats['mode'],
            'Jerk_Max': jerk_stats['max'],
            'Jerk_Min': jerk_stats['min'],
            'Jerk_Mean': jerk_stats['mean'],
            'Jerk_Median': jerk_stats['median'],
            'Jerk_Std': jerk_stats['std'],
            'Jerk_Mode': jerk_stats['mode']
        })
    sheet3_df = pd.DataFrame(deriv_stats_rows)
    
    # ---- Save per-subject Excel ----
    output_file = os.path.join(output_folder, f"{subject_id}_analysis.xlsx")
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        sheet1_df.to_excel(writer, sheet_name='TimeSeries', index=False)
        sheet2_df.to_excel(writer, sheet_name='AngleStats', index=False)
        sheet3_df.to_excel(writer, sheet_name='DerivativeStats', index=False)
    print(f"Saved {output_file}")

# ========================== CREATE GLOBAL SUMMARY EXCEL ==========================
if global_angle_stats:
    global_angle_df = pd.DataFrame(global_angle_stats)
    global_deriv_df = pd.DataFrame(global_deriv_stats)
    summary_file = os.path.join(output_folder, "All_Subjects_Summary.xlsx")
    with pd.ExcelWriter(summary_file, engine='openpyxl') as writer:
        global_angle_df.to_excel(writer, sheet_name='AngleStats', index=False)
        global_deriv_df.to_excel(writer, sheet_name='DerivativeStats', index=False)
    print(f"\nSaved global summary: {summary_file} with {len(global_angle_df)} trials")
else:
    print("\nNo data to create global summary.")

print("\nAll done!")

Found 150 files.

Processing subject: subject_001 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_001_analysis.xlsx

Processing subject: subject_002 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_002_analysis.xlsx

Processing subject: subject_003 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_003_analysis.xlsx

Processing subject: subject_004 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_004_analysis.xlsx

Processing subject: subject_005 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_005_analysis.xlsx

Processing subject: subject_006 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_006_analysis.xlsx

Processing subject: subject_007 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample\sample output\subject_007_analysis.xlsx

Processing subject: subject_008 (6 trials)
Saved C:\Users\admin\Downloads\Excel\sample